# Volume 8, Chapter 5: The Fundamental Group and Coverings (Python companion)

Fundamental groups of graphs computed from spanning trees, Stallings foldings for subgroups of free groups, and counting coverings with permutations.

*Before each solution, predict what the output will look like.*

## Exercises

**P5.1 (★) CS application: the cycle space of a network.** Theorem 5.23 says that $\pi_1$ of a connected graph is free on the edges outside a spanning tree. (a) Implement it: build a breadth-first spanning tree, and for each non-tree edge $e$ the loop $\gamma_e$. (b) Write a function that turns any closed walk at the base vertex into a reduced word in the generators $\gamma_e$ (read off the non-tree edges it crosses, with signs, and cancel). (c) Check on the Petersen graph that the rank is $E-V+1=6$, and that inserting random backtracks into a closed walk (a homotopy) never changes the reduced word, while different reduced words come from walks that are not homotopic. In circuit theory the same generators give Kirchhoff's independent loop equations, one per edge outside a spanning tree.

**P5.2 (★★) CS application: Stallings foldings.** Given words $w_1,\dots,w_k$ in $F_2=\langle a,b\rangle$, the subgroup $H=\langle w_1,\dots,w_k\rangle$ is the image of $\pi_1$ of a ``flower'': $k$ loops at a vertex, the $i$-th subdivided and labelled by $w_i$. Stallings (1983) observed that whenever two edges with the same label leave (or enter) the same vertex, they can be identified without changing the image subgroup; when no fold is possible, the graph is an immersion, and a word lies in $H$ exactly when it can be read as a closed path at the base vertex. (a) Implement folding and the membership test. (b) Show that $H$ has finite index exactly when the folded graph is a covering (every vertex has one incoming and one outgoing edge of each label), and then its number of vertices is the index. (c) Check the rank formula of Nielsen--Schreier on examples, including $\langle a^2,ab,b^2\rangle$ and $\langle a^3,b,aba^{-1},a^2ba^{-2}\rangle$, and test membership of some words.

**P5.3 (★★) Counting coverings.** A connected $n$-sheeted covering of the figure eight with a labelled base point is the same thing as a pair of permutations $(\sigma,\tau)$ of $\{1,\dots,n\}$ (how $a$ and $b$ move the sheets) generating a transitive group, modulo relabelling the sheets other than the base sheet $1$. (a) Deduce that the number $s_n$ of subgroups of index $n$ in $F_2$ is the number of transitive pairs divided by $(n-1)!$, and count them by brute force for $n\le5$. (b) Compare with Marshall Hall's recursion (1949), $s_n=n(n!)-\sum_{k=1}^{n-1}((n-k)!)\,s_k$, obtained by sorting all pairs by the size of the orbit of $1$. (c) How fast does $s_n$ grow? Compare $s_n$ with $n\cdot n!$.

## Solutions

### P5.1

In [ ]:
from collections import deque
import random

def petersen():
    E = [(i, (i + 1) % 5) for i in range(5)] + [(i, i + 5) for i in range(5)] + [(5 + i, 5 + (i + 2) % 5) for i in range(5)]
    return 10, E

def spanning_tree(nv, E, root=0):
    """breadth-first spanning tree; returns parent edge index of each vertex and the set of tree edges"""
    adj = [[] for _ in range(nv)]
    for k, (u, w) in enumerate(E):
        adj[u].append((w, k)); adj[w].append((u, k))
    parent, seen, q = {root: None}, {root}, deque([root])
    while q:
        u = q.popleft()
        for w, k in adj[u]:
            if w not in seen:
                seen.add(w); parent[w] = k; q.append(w)
    tree = {k for k in parent.values() if k is not None}
    return parent, tree, adj

def reduce(word):
    out = []
    for g in word:
        if out and out[-1] == (g[0], -g[1]):
            out.pop()
        else:
            out.append(g)
    return out

nv, E = petersen()
parent, tree, adj = spanning_tree(nv, E)
gens = [k for k in range(len(E)) if k not in tree]
print(f"V = {nv}, E = {len(E)}, tree edges = {len(tree)}, generators = {len(gens)} (E - V + 1 = {len(E) - nv + 1})")

def walk_to_word(walk):
    """walk = list of (edge index, direction +1 if traversed u->w); returns the reduced word in the generators"""
    return reduce([(k, d) for k, d in walk if k not in tree])

def random_closed_walk(length, start=0):
    # random walk, then return to the root along the tree
    v, walk = start, []
    for _ in range(length):
        w, k = random.choice(adj[v])
        walk.append((k, +1 if E[k][0] == v else -1)); v = w
    while v != start:
        k = parent[v]; u, w = E[k]
        nxt = u if w == v else w
        walk.append((k, +1 if E[k][0] == v else -1)); v = nxt
    return walk

def add_backtracks(walk, m):
    walk = list(walk)
    for _ in range(m):
        i = random.randrange(len(walk) + 1)
        v = 0
        for k, d in walk[:i]:
            v = E[k][1] if d == 1 else E[k][0]
        w, k = random.choice(adj[v])
        d = +1 if E[k][0] == v else -1
        walk[i:i] = [(k, d), (k, -d)]
    return walk

random.seed(3)
same = 0
for trial in range(1000):
    w = random_closed_walk(30)
    if walk_to_word(add_backtracks(w, 10)) == walk_to_word(w):
        same += 1
print(f"backtracks inserted in 1000 random closed walks: reduced word unchanged in {same} cases")
w = random_closed_walk(12)
print("a closed walk of length", len(w), "-> word", walk_to_word(w))

(a)--(b) A closed walk crosses tree edges and non-tree edges; collapsing the tree (which is contractible) leaves only the non-tree edges, each a loop $\gamma_e$, so the word of the walk is the sequence of non-tree edges it crosses, with orientation. Inserting a backtrack $e\,e^{-1}$ inserts $\gamma_e\gamma_e^{-1}$ or nothing, so the reduced word is a homotopy invariant, as the experiment confirms ($1000$ out of $1000$). By Theorem 5.23 it is a complete invariant: two closed walks are homotopic exactly when their reduced words agree.

(c) The Petersen graph has $V=10$, $E=15$, so $\pi_1$ is free of rank $6$. In an electrical network with $E$ wires and $V$ junctions, Kirchhoff's voltage law needs exactly $E-V+1$ independent loop equations, one for each edge outside a spanning tree: the abelianization of the same free group, $H_1=\Z^{E-V+1}$, which Chapter 6 calls the first homology group.

### P5.2

In [ ]:
class Folded:
    """a labelled graph with base vertex 0; edges stored as dicts out[v][x] = w, inn[w][x] = v for x in 'ab'"""
    def __init__(self, words):
        self.parent = {}
        self.edges = set()
        self.n = 1
        for w in words:
            v = 0
            for i, ch in enumerate(w):
                last = (i == len(w) - 1)
                nxt = 0 if last else self.new()
                x, inv = ch.lower(), ch.isupper()     # capital letter = inverse
                self.edges.add((nxt, x, v) if inv else (v, x, nxt))
                v = nxt
        self.fold()

    def new(self):
        self.n += 1
        return self.n - 1

    def find(self, v):
        while self.parent.get(v, v) != v:
            v = self.parent[v]
        return v

    def fold(self):
        changed = True
        while changed:
            changed = False
            E = {(self.find(u), x, self.find(w)) for u, x, w in self.edges}
            out, inn = {}, {}
            for u, x, w in sorted(E):
                if (u, x) in out and out[(u, x)] != w:
                    a, b = sorted((w, out[(u, x)])); self.parent[b] = a; changed = True; break
                if (w, x) in inn and inn[(w, x)] != u:
                    a, b = sorted((u, inn[(w, x)])); self.parent[b] = a; changed = True; break
                out[(u, x)] = w; inn[(w, x)] = u
            self.edges = E
        self.out = {(u, x): w for u, x, w in self.edges}
        self.inn = {(w, x): u for u, x, w in self.edges}
        self.V = {self.find(v) for v in range(self.n)}

    def member(self, word):
        v = 0
        for ch in word:
            x = ch.lower()
            key = (v, x)
            v = (self.inn if ch.isupper() else self.out).get(key)
            if v is None:
                return False
        return v == 0

    def is_covering(self):
        return all((v, x) in self.out and (v, x) in self.inn for v in self.V for x in "ab")

    def rank(self):
        return len(self.edges) - len(self.V) + 1

examples = {"<a^2, ab, b^2>": ["aa", "ab", "bb"],
            "<a^3, b, aba^-1, a^2ba^-2>": ["aaa", "b", "abA", "aabAA"],
            "<a, bab^-1>": ["a", "baB"],
            "<ab, ba>": ["ab", "ba"],
            "<a^2, b^2, ab, ba>": ["aa", "bb", "ab", "ba"]}
for name, ws in examples.items():
    G = Folded(ws)
    cov = G.is_covering()
    k = len(G.V)
    info = f"index {k}, Schreier 1 + k(n-1) = {1 + k}" if cov else "infinite index"
    print(f"{name:42s} vertices {k}, edges {len(G.edges)}, rank {G.rank()}, {info}")

G = Folded(["aa", "ab", "bb"])
for w in ["ba", "abab", "aab", "bAAb", "baB"]:
    print(f"   {w:6s} in <a^2, ab, b^2>?  {G.member(w)}")
H = Folded(["a", "baB"])
for w in ["bab", "baaB", "aba", "abaaBA"]:
    print(f"   {w:6s} in <a, bab^-1>?    {H.member(w)}")

(a) Folding never changes the image subgroup: identifying two edges with the same label at a vertex is a map of labelled graphs that is onto on $\pi_1$-images, because a path through the identified edge in the new graph lifts to a path in the old one up to inserting a backtrack $e\bar e'$, whose label cancels. When no fold is possible, the labelled graph maps to the figure eight by an *immersion* (locally injective), and reduced words are read along unique paths, so membership is exactly ``the reduced word is a closed path at the base''. This is a linear-time algorithm for the subgroup membership problem in free groups.

(b) If every vertex has all four half-edges, the folded graph is a covering of the figure eight, connected, with image $H$; its number of sheets is the index by Proposition 5.11. If some half-edge is missing, the graph can be completed into a covering only by adding infinite trees (``hanging'' the missing labels), so the index is infinite.

(c) The output shows rank $3$ at index $2$ for $\langle a^2,ab,b^2\rangle$ and rank $4$ at index $3$ for the second example, both $1+k$ as Nielsen--Schreier predicts for $n=2$. The subgroup $\langle a,bab^{-1}\rangle$ is free of rank $2$ with infinite index, as is $\langle ab,ba\rangle$; in the last example the four given generators fold to a graph of rank $3$, revealing one redundant generator ($ba=b^2(ab)^{-1}a^2$, read along the folded graph).

### P5.3

In [ ]:
import itertools, math

def transitive_pairs(n):
    perms = list(itertools.permutations(range(n)))
    count = 0
    for s in perms:
        for t in perms:
            seen, stack = {0}, [0]
            while stack:
                v = stack.pop()
                for w in (s[v], t[v]):
                    if w not in seen:
                        seen.add(w); stack.append(w)
            count += (len(seen) == n)
    return count

hall = {}
for n in range(1, 11):
    hall[n] = n * math.factorial(n) - sum(math.factorial(n - k) * hall[k] for k in range(1, n))

for n in range(1, 6):
    tp = transitive_pairs(n)
    print(f"n={n}: transitive pairs {tp:6d},  / (n-1)! = {tp // math.factorial(n-1):4d},  Hall: {hall[n]}")
for n in range(1, 11):
    print(f"n={n:2d}: s_n = {hall[n]:>12d},  s_n / (n n!) = {hall[n] / (n * math.factorial(n)):.5f}")

(a) A connected $n$-sheeted covering of the figure eight with its fibre over the vertex labelled $1,\dots,n$ (base sheet $1$) is determined by the two permutations of the fibre given by lifting $a$ and $b$, and conversely every pair of permutations defines a covering (the Schreier graph), connected exactly when the group they generate is transitive. Two labellings with the same base sheet give isomorphic based coverings, and conversely an isomorphism of based coverings is a relabelling fixing $1$; the group of such relabellings, of order $(n-1)!$, acts freely on transitive pairs. By Theorem 5.16, based connected coverings correspond to subgroups, and the subgroup has index $n$ by Proposition 5.11. The brute-force counts agree with Hall: $1,3,13,71,461$.

(b) Among all $(n!)^2$ pairs, sort by the orbit of $1$: if it has size $k$, choose the other $k-1$ points of the orbit, a transitive pair on it ($s_k(k-1)!$ ways), and any pair on the remaining $n-k$ points ($((n-k)!)^2$ ways). So $(n!)^2=\sum_k\binom{n-1}{k-1}(k-1)!\,s_k\,((n-k)!)^2$, which rearranges to Hall's formula.

(c) The ratio $s_n/(n\cdot n!)$ creeps up towards $1$, roughly like $1-\frac1n$ ($0.88$ at $n=10$): almost every pair of random permutations generates a transitive group (in fact, by a theorem of Dixon (1969), the whole symmetric or alternating group), so $s_n\sim n\cdot n!$. A free group of rank $2$ has more subgroups of index $n$ than there are permutations of $n$ objects.